## Recomendación: Funciones

Constantes, métricas de ranking y funciones compartidas

### Installs

`faiss-cpu`: índices de vecinos aproximados (IVF y HNSW)

In [ ]:
%pip install -q faiss-cpu==1.8.0

### Imports

In [ ]:
import json
import numpy as np
import scipy.sparse as sp

### Constantes

- `SEED` = 42: misma semilla en muestras y modelos

- `PARTICIONES`: años de entrenamiento y de evaluación de cada partición

    - `val`: entrenar 2023 → evaluar 2024; se eligen los parámetros

    - `test`: entrenar 2023-2024 → evaluar 2025; se usa una vez por modelo

- `K_EVAL` = 10: largo de la lista que se evalúa

- `N_CAND` = 100: candidatos guardados por entidad y modelo, para el híbrido

In [ ]:
SEED = 42
PARTICIONES = {"val": ([2023], 2024), "test": ([2023, 2024], 2025)}
K_EVAL = 10
N_CAND = 100
datos_dir = Path.cwd() / "temp" / "datos"
datos_dir.mkdir(parents=True, exist_ok=True)

### Parámetros

| Parámetro | Valor | Por qué |
|---|---|---|
| `MIN_ENTIDADES` | 2 | Conserva el 74% de los positivos y saca a los consorcios de un solo uso |
| `SENAL` | binaria | El 81% de los pares tiene 1 adjudicación |
| `K_VEC` | users: 100, items: 400 | Máximo NDCG en validación; con 50 vecinos casi no cambia |
| `MIN_DF`, `MAX_DF` | 5, 0.5 | Vocabulario de ~12.6k términos; > 50% de procesos no distingue rubros |
| `DIM_SVD` | 512 | Máximo NDCG en validación (0.053 vs 0.047 con 256) |
| `ANN` | M, efSrch = 32, 256 | Más rápido con recall@100 ≥ 0.95: 0.50 ms por consulta vs 2.83 ms exacto |
| `RANK`, `REG`, `ALPHA` | 64, 1.0, 40 | Máximo NDCG en validación; rank 128 tarda 4× y no mejora |
| `W_HIBRIDO` | 0.7 | Máximo NDCG en validación (0.115 vs 0.111 de ALS solo) |
| `UMBRAL_HHI` | 0.25 | Umbral usual de alta concentración |
| `P_INUSUAL` | 5 | 4,805 relaciones; el perfil es estable entre 5% y 10% |

In [ ]:
MIN_ENTIDADES = 2
SENAL = "binaria"
K_VEC = {"usuario": 100, "item": 400}
MIN_DF, MAX_DF = 5, 0.5
DIM_SVD = 512
ANN = {"tipo": "hnsw", "M": 32, "ef": 256}
RANK, REG, ALPHA = 64, 1.0, 40
W_HIBRIDO = 0.7
UMBRAL_HHI = 0.25
P_INUSUAL = 5

### Configuración notebooks

`config.json` guarda las decisiones

In [ ]:
def leer_config():
    ruta = datos_dir / "config.json"
    return json.loads(ruta.read_text()) if ruta.exists() else {}

def guardar_config(config):
    (datos_dir / "config.json").write_text(json.dumps(config, indent=1, ensure_ascii=False))

### Particiones

- `cargar_particion`: pares de entrenamiento y positivos de evaluación

- `matriz`: matriz dispersa entidad × proveedor con su señal

In [ ]:
SENALES = {"binaria": lambda d: np.ones(len(d)), "n_adj": lambda d: np.log1p(d.n_adj.to_numpy()),
           "log_monto": lambda d: d.senal_log_monto.to_numpy()}

def cargar_particion(p):
    tr = pd.read_parquet(datos_dir / f"{p}_train")
    ev = pd.read_parquet(datos_dir / f"{p}_eval")
    pos = ev[ev.tipo == "nuevo"].groupby("entidad_idx").proveedor_idx.apply(set).to_dict()
    return tr, ev, pos

def matriz(tr, senal, forma):
    return sp.csr_matrix((SENALES[senal](tr).astype(np.float32), (tr.entidad_idx, tr.proveedor_idx)), shape=forma)

### Ranking y métricas

`Evaluación`: De entidad a proveedor

- `top_k`: puntúa por lotes de entidades

- **NDCG**: premia los aciertos arriba en la lista; 1 = todos los positivos primero

- **HitRate**: entidades con al menos un acierto en el top-k

- **Cobertura**: proveedores distintos recomendados / tamaño del catálogo

In [ ]:
def top_k(puntaje, entidades, vistos, catalogo, k=N_CAND, lote=256):
    filas = []
    fuera = ~catalogo
    for i in range(0, len(entidades), lote):
        idx = np.asarray(entidades[i:i + lote])
        s = np.asarray(puntaje(idx), dtype=np.float32)
        s[:, fuera] = -np.inf
        v = vistos[idx].tocoo()
        s[v.row, v.col] = -np.inf
        top = np.argpartition(-s, k, axis=1)[:, :k]
        orden = np.take_along_axis(top, np.argsort(-np.take_along_axis(s, top, 1), axis=1), 1)
        sc = np.take_along_axis(s, orden, 1)
        for e, items, scores in zip(idx, orden, sc):
            ok = np.isfinite(scores)
            filas.append(pd.DataFrame({"entidad_idx": e, "rank": np.arange(ok.sum()) + 1,
                                       "proveedor_idx": items[ok], "score": scores[ok]}))
    return pd.concat(filas, ignore_index=True)

def evaluar(recs, pos, n_catalogo, k=K_EVAL):
    r = recs[recs["rank"] <= k]
    listas = r.groupby("entidad_idx").proveedor_idx.apply(list).to_dict()
    disc = 1 / np.log2(np.arange(2, k + 2))
    p, rc, nd, hit = [], [], [], []
    for e, verdad in pos.items():
        lista = listas.get(e, [])
        acierto = np.array([x in verdad for x in lista], dtype=float)
        n = acierto.sum()
        p.append(n / k); rc.append(n / len(verdad)); hit.append(float(n > 0))
        ideal = disc[:min(len(verdad), k)].sum()
        nd.append((acierto * disc[:len(acierto)]).sum() / ideal)
    return {f"precision@{k}": round(np.mean(p), 4), f"recall@{k}": round(np.mean(rc), 4), f"ndcg@{k}": round(np.mean(nd), 4),
            f"hit@{k}": round(np.mean(hit), 4), "cobertura": round(r.proveedor_idx.nunique() / n_catalogo, 4)}

def guardar_recs(recs, nombre):
    recs.to_parquet(datos_dir / f"recs_{nombre}", index=False)

def leer_recs(nombre):
    return pd.read_parquet(datos_dir / f"recs_{nombre}")

### Contexto de una partición

Todo lo que un modelo necesita para puntuar y evaluar una partición

In [ ]:
def contexto(p, senal="binaria"):
    cfg = leer_config()
    tr, ev, pos = cargar_particion(p)
    forma = (cfg["n_entidades"], cfg["n_proveedores"])
    X = matriz(tr, senal, forma)
    grado = np.asarray((X > 0).sum(0)).ravel()
    catalogo = grado >= cfg.get("min_entidades", MIN_ENTIDADES)
    entidades = np.array(sorted(pos))
    return {"tr": tr, "ev": ev, "pos": pos, "X": X, "vistos": (X > 0).astype(np.int8).tocsr(),
            "catalogo": catalogo, "entidades": entidades, "forma": forma}

### Vecindario user-user

- Coseno entre entidades: cada entidad se queda con sus `k` vecinas más parecidas

- Puntaje(e, p) = Σ similitud(e, v) · señal(v, p) sobre las vecinas v

In [ ]:
def vecinas(X, k):
    Xn = sp.csr_matrix(X.multiply(1 / np.sqrt(X.multiply(X).sum(1)).clip(1e-9)))
    S = (Xn @ Xn.T).toarray()
    np.fill_diagonal(S, 0)
    if k < S.shape[0]:
        corte = np.argpartition(-S, k, axis=1)[:, k:]
        np.put_along_axis(S, corte, 0, axis=1)
    return sp.csr_matrix(S)

def puntaje_usuario(X, k):
    S = vecinas(X, k)
    return lambda idx: (S[idx] @ X).toarray()

### Normalización

Min-max del puntaje dentro de cada entidad, para sumar modelos con escalas distintas

In [ ]:
def normalizar(recs):
    g = recs.groupby("entidad_idx").score
    mn, mx = g.transform("min"), g.transform("max")
    return recs.assign(score=((recs.score - mn) / (mx - mn).replace(0, 1)).fillna(1.0))